In [1]:
import datetime
import os
import uuid

from dotenv import load_dotenv
import xarray as xr
from xsdba.adjustment import QuantileDeltaMapping, TrainAdjust
from xsdba.base import Grouper

In [35]:
#load_dotenv()

FORECAST_URI = "/home/emily_zuetell/projects/poreallas/data/parsed/08_forecast_parsed.zarr"
GMFD_URI = "/home/emily_zuetell/projects/poreallas/data/parsed/gmfd_parsed.zarr"
OUT_ZARR = "/home/emily_zuetell/projects/poreallas/data/forecast_adj.zarr"
HISTREF_START_YEAR = 1981
HISTREF_STOP_YEAR = 1997
SIM_START_YEAR = 2007
SIM_STOP_YEAR = 2027
QDM_N_QUANTILES = 10
FORECAST_LENGTH = 215  # ECMWF S51 is 215 days.
UID = str(uuid.uuid4())
START_TIME = datetime.datetime.now(datetime.UTC).isoformat()



In [36]:
def adjust_month(
    *,
    ref: xr.DataArray,
    hist: xr.DataArray,
    sim: xr.DataArray,
    target_month: int,
    nquantiles: int,
) -> tuple[TrainAdjust, xr.DataArray]:
    """
    Train and apply QDM for a particular `time.month`
    """
    ref = ref.where(ref["time.month"] == target_month, drop=True)
    hist = hist.where(hist["time.month"] == target_month, drop=True)
    sim = sim.where(sim["time.month"] == target_month, drop=True)

    # Check for rollover months, reduce years in ref
    ref = ref.where(ref["time"].isin(hist["time"]), drop=True)

    qdm = QuantileDeltaMapping.train(
        ref,
        hist,
        nquantiles=nquantiles,
        kind="+",
        group=Grouper("time", add_dims=["number"]),
    )
    adj = qdm.adjust(sim)
    sim_adj = adj.isel(time=slice(-int(FORECAST_LENGTH), None))

    sim_adj.name = "tas"
    sim_adj = sim_adj.to_dataset()

    # Add additional general metadata.
    sim_adj.attrs |= {
        "poreallas_created_at": START_TIME,
        "poreallas_uid": UID,
        "poreallas_description": "QDM bias-adjusted forecast ensemble fields",
    }
    sim_adj["tas"].attrs |= {
        "poreallas_created_at": START_TIME,
        "poreallas_uid": UID,
        "poreallas_description": "QDM bias-adjusted forecast ensemble tas fields",
        "poreallas_adjustment_method": "QDM",
        "poreallas_histref_start_year": HISTREF_START_YEAR,
        "poreallas_histref_stop_year": HISTREF_STOP_YEAR,
        "poreallas_sim_start_year": SIM_START_YEAR,
        "poreallas_sim_stop_year": SIM_STOP_YEAR,
        "poreallas_qdm_nquantiles": QDM_N_QUANTILES,
        "poreallas_ref_uri": GMFD_URI,
        "poreallas_hist_uri": FORECAST_URI,
        "poreallas_sim_uri": FORECAST_URI,
    }

    sim_adj = sim_adj.chunk("auto")

    sim_adj.to_zarr(f"/home/emily_zuetell/projects/poreallas/data/m{target_month}_forecast_adj.zarr", consolidated=True)

    return qdm, adj


def adjust_months(
    *,
    ref: xr.DataArray,
    hist: xr.DataArray,
    sim: xr.DataArray,
    nquantiles: int,
) -> xr.DataArray:
    """
    Train and apply quantile delta mapping (QDM) for all `time.month` in a simulation.

    We need a custom algorithm for this because our forecast ensembles run for <
    365 days yet this QDM implementation does not allow us to group by "time.month"
    when it does not have all 12 months. So we train and apply QDM to each of the
    months in the simulation dataset and then concatenate them back together along
    the time dimension. The concatenated data is then sorted by the time dimension
    to return the data to chronological order.

    Parameters
    ----------
    ref :
        Reference dataset to compare against a historical simulation to train a QDM.
    hist :
        Historical simulation dataset to be compared against ref when training the QDM.
    sim :
        Simulation, or forecast ensemble to be adjusted by the trained QDM.
    nquantiles :
        Number of quantiles to use in the quantile mapping.

    Returns
    -------
    combined :
        Simulated, bias-adjusted by a QDM trained on a historical and reference dataset.
    """
    adjusted = []
    for m in set(sim["time.month"].data):
        _, adj = adjust_month(
            ref=ref,
            hist=hist,
            sim=sim,
            target_month=m,
            nquantiles=nquantiles,
        )
        adjusted.append(adj)

    combined = xr.concat(adjusted, dim="time").sortby("time")
    return combined

In [37]:
gmfd = xr.open_zarr(GMFD_URI)
forecast = xr.open_zarr(FORECAST_URI)

In [38]:
# Outline the datasets we need for the adjustment, grabbing the windows in time needed.
ref = gmfd.sel(time=slice(str(HISTREF_START_YEAR), str(HISTREF_STOP_YEAR)))
hist = forecast.sel(time=slice(str(HISTREF_START_YEAR), str(HISTREF_STOP_YEAR)))
sim = forecast.sel(time=slice(str(SIM_START_YEAR), str(SIM_STOP_YEAR)))

In [39]:
# # Subset reference to only daysofyear that are in our forecast ensemble. The
# forecast ensemble has incomplete years. Ref/hist/sim need to have matching
# ragged ends in their time series for QDM.
ref = ref.where(ref["time.dayofyear"].isin(sim["time.dayofyear"]), drop=True)
hist = hist.where(hist["time.dayofyear"].isin(sim["time.dayofyear"]), drop=True)

# Rechunking because all of "time", or whatever we're grouping QDM on, needs to be in one chunk.
ref = ref.chunk({"time": -1})
hist = hist.chunk({"number": -1, "time": -1, "latitude": "30", "longitude": "auto"})
sim = sim.chunk({"number": -1, "time": -1, "latitude": "30", "longitude": "auto"})


In [40]:
target_month = 1
ref=ref["tas"]
hist=hist["tas"]
sim=sim["tas"]
nquantiles=QDM_N_QUANTILES

ref = ref.where(ref["time.month"] == target_month, drop=True)
hist = hist.where(hist["time.month"] == target_month, drop=True)
sim = sim.where(sim["time.month"] == target_month, drop=True)

# Check for rollover months, reduce years in ref
ref = ref.where(ref["time"].isin(hist["time"]), drop=True)

qdm = QuantileDeltaMapping.train(
    ref,
    hist,
    nquantiles=nquantiles,
    kind="+",
    group=Grouper("time", add_dims=["number"]),
)



In [41]:
adj = qdm.adjust(sim.where(sim["time.year"] == 2027, drop=True))

In [42]:
adj

<xarray.DataArray 'scen' (time: 31, longitude: 360, latitude: 180, number: 51)> Size: 410MB
dask.array<scen-block_qdm_adjust, shape=(31, 360, 180, 51), dtype=float32, chunksize=(31, 1, 1, 51), chunktype=numpy.ndarray>
Coordinates:
  * time                     (time) object 248B 2027-01-01 00:00:00 ... 2027-...
    forecast_period          (time) timedelta64[ns] 248B dask.array<chunksize=(31,), meta=np.ndarray>
    forecast_reference_time  (time) datetime64[ns] 248B dask.array<chunksize=(31,), meta=np.ndarray>
  * longitude                (longitude) float64 3kB 0.5 1.5 2.5 ... 358.5 359.5
  * latitude                 (latitude) float64 1kB 89.5 88.5 ... -88.5 -89.5
  * number                   (number) int64 408B 0 1 2 3 4 5 ... 46 47 48 49 50
Attributes: (12/33)
    GRIB_dataType:                            fc
    GRIB_numberOfPoints:                      64800
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    GRIB_gridType:                            regular_ll
    ...                                       ...
    GRIB_surface:                             0.0
    poreallas_created_at:                     2026-08-06T16:07:16.417760+00:00
    poreallas_uid:                            5fa02d07-3f21-4c51-81e1-eef50b7...
    poreallas_description:                    Parsed ECMWF S51 ensemble tas f...
    history:                                  [2026-08-06 18:07:35] : Bias-ad...
    bias_adjustment:                          QuantileDeltaMapping(group=Grou...

In [43]:
sim_adj = adj

sim_adj.name = "tas"
sim_adj = sim_adj.to_dataset()

# Add additional general metadata.
sim_adj.attrs |= {
    "poreallas_created_at": START_TIME,
    "poreallas_uid": UID,
    "poreallas_description": "QDM bias-adjusted forecast ensemble fields",
}
sim_adj["tas"].attrs |= {
    "poreallas_created_at": START_TIME,
    "poreallas_uid": UID,
    "poreallas_description": "QDM bias-adjusted forecast ensemble tas fields",
    "poreallas_adjustment_method": "QDM",
    "poreallas_histref_start_year": HISTREF_START_YEAR,
    "poreallas_histref_stop_year": HISTREF_STOP_YEAR,
    "poreallas_sim_start_year": SIM_START_YEAR,
    "poreallas_sim_stop_year": SIM_STOP_YEAR,
    "poreallas_qdm_nquantiles": QDM_N_QUANTILES,
    "poreallas_ref_uri": GMFD_URI,
    "poreallas_hist_uri": FORECAST_URI,
    "poreallas_sim_uri": FORECAST_URI,
}

sim_adj = sim_adj.chunk("auto")

sim_adj.to_zarr(f"/home/emily_zuetell/projects/poreallas/data/m{target_month}_forecast_adj.zarr", consolidated=True)

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [48]:
import re
import glob

def month_num(path):
    return int(re.search(r"m(\d+)_forecast_adj\.zarr", path).group(1))

files = sorted(glob.glob("/home/emily_zuetell/projects/poreallas/data/m*_forecast_adj.zarr"), key=month_num)
ds = xr.open_mfdataset(files, engine="zarr", combine="nested", concat_dim="time")
for var in ds.variables:
    ds[var].encoding.pop("chunks", None)
ds = ds.chunk({"time": -1, "latitude": -1, "longitude": -1})
ds.to_zarr("/home/emily_zuetell/projects/poreallas/data/forecast_adj.zarr", mode="w")

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
